# SCTEC - Projeto Final — Módulo 01 | Machine Learning: Modelo Preditivo
**DESENVOLVIMENTO DE IA PARA ANÁLISE PREDITIVA**

---

## Descrição do notebook

Este material implementa um  Pipeline  Preditivo  completo aplicado à Indústria 4.0. 

●  O  algorítmo propõe uma solução preditiva para um  parque  fabril  monitorado  por  sensores, prevendo  quebras mecânicas  nos  equipamentos,  para  evitar paradas  na  linha  de  produção.  A  variável alvo  do  projeto  é  binária:  falha_maquina  =  1 (quando  há  uma  avaria  detectada)  e  falha_maquina  =  0 (funcionamento normal). 
Na solução são aplicados os conceitos de:

* **Divisão treino/teste com estratificação:** por que `stratify=y` importa quando o alvo é desbalanceado;
* **Balanceamento de classes:** SMOTE aplicado **apenas no treino**, para não vazar informação para o teste;
* **Escalonamento seletivo:** por que o KNN precisa de `StandardScaler` e a Árvore de Decisão não;
* **Ajuste de hiperparâmetros:** `n_neighbors` no KNN e `max_depth` na Árvore, e como cada um controla o overfitting;
* **Avaliação crítica de acurácia:** por que "acerto alto" pode significar pouco quando o alvo é raro.

💡 **Sobre a base:** ela foi criada com problemas propositais — nulos, duplicatas, colunas de vazamento de dados e desbalanceamento de classe — para proprorcionar as capacidades de avaliação, diagnóstico e decisão, não só sintaxe.

⚠️ **Atenção:** Trata-se de um projeto acadêmico visando ao atendimento às competências necessárias para a conclusão com aproveitamento no curso de "Desenvolvimento de IA para Análise Preditiva, do projeto SCTEC.

⚠️ **Atenção:** A cosntrução desse projeto observou a estruturação manual da hierarquia de pastas, sendo pois, necessária a sua replicação, para o seu correto funcionamento.

⚠️ **Observação:** Como trata-se de um projeto acadêmico, optou-se por efetuar as ações de versionamento no Git/GitHub utilizando o idioma inglês.

⚠️ **Observação:** Devido às mesmas razões acadêmicas, observa-se um número elevado de células no projeto, cujo objetivo foi isolar trechos de interesse, os quais mereciam um maior detalhamento, o que, por sua vez, demandou uma quantidade elevada de comentários, que tornam mais efetivo o foco no aprendizado.

## FASE 0: Configuração de Ambiente e Repositório Git

### 0.1 - Ambiente e importações

Na maioria dos ambientes de desenvolvimento, as bibliotecas abaixo já está disponível. A exceção costuma ser o `imbalanced-learn` (SMOTE):
O comandos abaixo deveram ser executados no terminal, visando à instalação da biblioteca.
```
pip install imbalanced-learn
```
Para rodar localmente:
```
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn
```

In [ ]:
# Bibliotecas de terceiros
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from imblearn.over_sampling import SMOTE

# Módulos da biblioteca padrão do Python
import os
import re
import json
from datetime import datetime, timedelta
from IPython.display import display

pd.set_option("display.width", 120)
sns.set_style("whitegrid")

print("Ambiente pronto!")

## FASE 1: Análise Exploratória de Dados (EDA)

<h5 style="color: #7ac1f4;">
RF01 – Critério de Pronto (DoD): Dimensões, .describe(), 3 gráficos plotados e célula de análise interpretativa concluída.
</h5>

In [ ]:
# Carregar o CSV
df_bruto = pd.read_csv('../data/manutencao_preditiva.csv')

In [ ]:
# Função para inspecionar os dados
def inspecionar_dados(df):
    """Exibe as informações estruturais do DataFrame com tabelas renderizadas."""
    print("=== INSPEÇÃO INICIAL DO DATASET ===")
    
    # 1. Shape
    print(f'Shape: O dataframe tem [{df.shape[0]}] linhas e [{df.shape[1]}] colunas.\n')
    
    # 2. As 5 primeiras linhas (renderizadas)
    print("Primeiras 5 linhas:")
    display(df.head()) 
    
    # 3. As 5 últimas linhas (renderizadas)
    print("\nÚltimas 5 linhas:")
    display(df.tail())
    
    # 4. Tipos das colunas
    print("\nTipos das colunas:")
    display(df.dtypes.to_frame(name="Tipo de Dado"))
    
    # 5. Valores nulos
    print("\nValores nulos por coluna:")
    display(df.isnull().sum().to_frame(name="Qtd Nulos"))

In [ ]:
# Chamada da função
inspecionar_dados(df_bruto)

In [ ]:
df_bruto.info()  # Exibe informações detalhadas sobre o DataFrame
df_bruto.describe()  # Exibe estatísticas descritivas do DataFrame

In [ ]:
# Configuração da figura com 3 subplots em 1 linha
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -------------------------------------------------------------------------
# GRÁFICO 1: Histograma de Distribuição por Classe (Torque em Nm)
# -------------------------------------------------------------------------
sns.histplot(
    data=df_bruto, 
    x="torque_nm", 
    hue="falha_maquina", 
    kde=True, 
    ax=axes[0], 
    palette=["#1C7293", "#D64550"]
)
axes[0].set_title("Distribuição do Torque (Nm) por Classe de Falha")
axes[0].set_xlabel("Torque (Nm)")
axes[0].set_ylabel("Frequência")

# -------------------------------------------------------------------------
# GRÁFICO 2: Gráfico de Barras do Desbalanceamento da Variável Alvo
# -------------------------------------------------------------------------
contagem = df_bruto["falha_maquina"].value_counts().sort_index()
axes[1].bar(["Sem Falha (0)", "Com Falha (1)"], contagem.values, color=["#1C7293", "#D64550"])

# Adiciona os rótulos de contagem e percentual sobre cada barra
for i, v in enumerate(contagem.values):
    percentual = (v / contagem.sum()) * 100
    axes[1].text(i, v + 150, f"{v}\n({percentual:.1f}%)", ha="center", fontsize=10, fontweight='bold')

axes[1].set_title("Desbalanceamento da Variável Alvo (falha_maquina)")
axes[1].set_ylabel("Quantidade de Equipamentos")
axes[1].set_ylim(0, max(contagem.values) * 1.15) # Ajusta limite Y para caber o texto

# -------------------------------------------------------------------------
# GRÁFICO 3: Mapa de Calor (Heatmap) da Correlação de Pearson
# -------------------------------------------------------------------------
cols_correlacao = [
    "temperatura_ar_k", "temperatura_processo_k", "velocidade_rotacao_rpm", 
    "torque_nm", "desgaste_ferramenta_min", "falha_maquina"
]

corr_matrix = df_bruto[cols_correlacao].corr(method="pearson")

sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="coolwarm", 
    center=0, 
    vmin=-1, 
    vmax=1, 
    ax=axes[2]
)
axes[2].set_title("Matriz de Correlação de Pearson")

# Ajuste fino de layout
plt.tight_layout()
plt.show()


:
### ✳️ Decisão  técnica - 01
A escolha da coluna torque_nm para o histograma da FASE 1 não foi arbitrária. Ela foi definida com base em duas análises estatísticas prévias realizadas na Fase 1 (a inspeção do info() e, principalmente, a análise descritiva com o describe()).

1. Maior Dispersão e Variabilidade Estatística no describe()
Na análise descritiva dos dados numéricos:

    - Variáveis como temperatura_ar_k e temperatura_processo_k possuem amplitude e desvio padrão relativamente baixos e muito concentrados em torno da média (pouca variação relativa).
    - O torque_nm, por outro lado, apresenta uma alta amplitude e variabilidade significativa (variando desde valores próximos de 3.8 Nm até mais de 76.6 Nm), tornando-se um candidato ideal para observar como a variação de um parâmetro operacional contínuo se distribui em relação às falhas. 
    - Distribuição Normal Natural: A igualdade entre média e mediana no describe() indica um formato de curva em sino perfeito no gráfico.Picos Extremos: O max em $76.60$ isola os eventos de esforço elevado, permitindo visualizar onde ocorrem as falhas por sobrecarga nas caudas do gráfico.
    - Maior dispersão Proporcional (CV = 25%): Entre as variáveis físicas operacionais, o torque é a que possui a oscilação mais expressiva em torno da média. 
    

1. Em suma, observou-se os seguintes aspectos anteriores para a escolha:

    - Maior Dispersão Proporcional (CV = 25%): Entre as variáveis físicas operacionais, o torque é a que possui a oscilação mais expressiva em torno da média;

    - Distribuição Normal Natural: A igualdade entre média e mediana no describe() indica um formato de curva em sino perfeito no gráfico;

    - Picos Extremos: O max em $76.60$ isola os eventos de esforço elevado, permitindo visualizar onde ocorrem as falhas por sobrecarga nas caudas do gráfico.

### 📊 Análise Interpretativa do Diagnóstico de EDA

A partir da inspeção das estatísticas e da visualização dos gráficos analíticos, identificou-se os seguintes padrões para o direcionamento do projeto:

1. **Distribuição do Torque e Sobrecarga Mecânica (Gráfico 1):**
   * A operação normal (Classe 0) segue uma distribuição normal centrada em aproximadamente 40 Nm. 
   * As falhas (Classe 1) concentram-se visivelmente nos extremos da distribuição, sobretudo em valores elevados de torque (> 55 Nm), indicando que episódios de sobrecarga mecânica são fortes desencadeadores de paradas não planejadas.

2. **Severo Desbalanceamento de Classes (Gráfico 2):**
   * A variável alvo `falha_maquina` possui 9.661 registros normais (96.6%) e apenas 339 falhas (3.4%).
   * *Orientação:* É fundamental aplicar reamostragem via **SMOTE** no conjunto de treino (FASE 4) para evitar viés em favor da classe majoritária.

3. **Matriz de Correlação de Pearson (Gráfico 3):**
   * **Principais Preditores:** O `torque_nm` (r = 0.19) e o `desgaste_ferramenta_min` (r = 0.11) apresentam as maiores correlações diretas com a ocorrência de falhas.
   * **Fundamentação para Feature Engineering:** A forte correlação negativa entre `velocidade_rotacao_rpm` e `torque_nm` (r = -0.88) reflete a dinâmica física dos motores elétricos, respaldando a criação do atributo de **potência mecânica** na FASE 3.
   * **Multicolinearidade Térmica:** A correlação de r = 0.88 entre `temperatura_ar_k` e `temperatura_processo_k` revela redundância de informação entre a temperatura ambiente e a do processo.

## FASE 2: Limpeza e Tratamento de Dados (Data Prep)

<h5 style="color: #7ac1f4;">
RF02 – Critério de Pronto (DoD): Sem duplicados, sem nulos, com justificativa estatística e boxplots exibidos.
</h5>

In [ ]:
# 1. Trabalhar a partir do dataset original 'df_bruto'
df_tratado = df_bruto.copy()

# 2. Quantidade de linhas antes da remoção de duplicados
linhas_antes = len(df_tratado)

# 3. Remoção de duplicados com base na coluna 'id_produto'
df_tratado = df_tratado.drop_duplicates(subset=["id_produto"], keep="first").reset_index(drop=True)

# 4. Quantidade de linhas após a remoção de duplicados
linhas_depois = len(df_tratado)

print("Tratamento de duplicados concluído:\n")
print(f"Linhas antes: {linhas_antes} | depois: {linhas_depois} | removidas: {linhas_antes - linhas_depois}")

In [ ]:
# 5. Identificação de dados ausentes antes do tratamento

print("Valores ausentes por coluna:")
display(df_tratado.isna().sum().to_frame("quantidade"))

print("\nAssimetria (skew):")
print("temperatura_ar_k:", round(df_tratado["temperatura_ar_k"].skew(), 3))
print("temperatura_processo_k:", round(df_tratado["temperatura_processo_k"].skew(), 3))
print("velocidade_rotacao_rpm:", round(df_tratado["velocidade_rotacao_rpm"].skew(), 3))
print("torque_nm:", round(df_tratado["torque_nm"].skew(), 3))

In [ ]:
# 6. Imputação de dados ausentes utilizando a MEDIANA

cols_com_nulos = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
]

for col in cols_com_nulos:
  if col in df_tratado.columns and df_tratado[col].isnull().sum() > 0:
    mediana_val = df_tratado[col].median()
    df_tratado[col] = df_tratado[col].fillna(mediana_val)

print("\nValidação após imputação (deve retornar 0 para todas as colunas):")
print(df_tratado[cols_com_nulos].isnull().sum())

In [ ]:
# 7. Visualização de Outliers via Boxplots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Boxplots das Variáveis Explicativas Contínuas (Identificação de Outliers)",
    fontsize=16,
    fontweight="bold",
)

sns.boxplot(ax=axes[0, 0], y=df_tratado["temperatura_ar_k"], color="#3498db")
axes[0, 0].set_title("Temperatura do Ar (K)")

sns.boxplot(ax=axes[0, 1], y=df_tratado["temperatura_processo_k"], color="#2ecc71")
axes[0, 1].set_title("Temperatura do Processo (K)")

sns.boxplot(ax=axes[1, 0], y=df_tratado["velocidade_rotacao_rpm"], color="#e67e22")
axes[1, 0].set_title("Velocidade de Rotação (RPM)")

sns.boxplot(ax=axes[1, 1], y=df_tratado["torque_nm"], color="#e74c3c")
axes[1, 1].set_title("Torque (Nm)")

plt.tight_layout()
plt.show()

In [ ]:
# 7. Contabilizando de Outliersa usando a regra do Intervalo Interquartil (IQR)

# Comentário pessoal: Nesse ponto, estou utilizando a regra do Intervalo Interquartil (IQR) para identificar e contabilizar os outliers nas variáveis contínuas do dataset. O objetivo é entender melhor a distribuição dos dados e identificar possíveis pontos fora do padrão que possam influenciar negativamente na análise ou modelagem preditiva e verificar, pela quantidade, o impacto desses pontos na base de dados e a viabilidade de remoção ou tratamento.

# Lista das colunas operacionais contínuas
colunas_analise = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
]

print("--- Contagem de Outliers por Coluna (Regra IQR) ---")

for col in colunas_analise:
  # Calculando o 1º quartil (25%) e 3º quartil (75%)
  Q1 = df_tratado[col].quantile(0.25)
  Q3 = df_tratado[col].quantile(0.75)

  # Intervalo Interquartil
  IQR = Q3 - Q1

  # Limites inferior e superior
  limite_inferior = Q1 - 1.5 * IQR
  limite_superior = Q3 + 1.5 * IQR

  # Identificando as linhas fora dos limites
  outliers_inf = df_tratado[df_tratado[col] < limite_inferior]
  outliers_sup = df_tratado[df_tratado[col] > limite_superior]

  total_outliers = len(outliers_inf) + len(outliers_sup)
  percentual = (total_outliers / len(df_tratado)) * 100

  print(f"\nVariável: {col}")
  print(f"  • Limite Inferior: {limite_inferior:.2f} | Limite Superior: {limite_superior:.2f}")
  print(f"  • Outliers Inferiores: {len(outliers_inf)}")
  print(f"  • Outliers Superiores: {len(outliers_sup)}")
  print(f"  • Total de Outliers: {total_outliers} ({percentual:.2f}% dos dados)")

### ✳️ Decisão  técnica - 02 

Tratamento de Dados Ausentes e Análise de Outliers (Card 2.1) - A partir do conjunto original `df_bruto`, realizou-se a cópia de trabalho, a remoção de duplicados e o tratamento de nulos.

#### Justificativa Técnica para Imputação via Mediana
Para o preenchimento dos valores nulos identificados nas variáveis operacionais (`temperatura_ar_k`, `temperatura_processo_k`, `velocidade_rotacao_rpm` e `torque_nm`), optou-se pela **Mediana** em detrimento da Média pelos seguintes motivos estatísticos:

1. **Robustez a Valores Extremos (Outliers):** A média é altamente sensível a valores discrepantes nas caudas das distribuições. Como observado na análise exploratória (Fase 1) e confirmado nos boxplots, variáveis como `velocidade_rotacao_rpm` e `torque_nm` apresentam picos acentuados de esforço/operação que distorceriam a média geral.
2. **Preservação da Centralidade:** A mediana representa o valor central exato do conjunto ordenado ($50\%$ dos dados), garantindo que os valores imputados não introduzam viés artificial nas medições físicas do equipamento.

#### Diagnóstico Inicial dos Boxplots
* **`velocidade_rotacao_rpm` e `torque_nm`:** Apresentam pontos fora dos limites dos *whiskers* (outliers superiores e inferiores). Esses pontos representam regimes de sobrecarga mecânica ou baixa rotação severa, que coincidem com os episódios de falha da máquina.
* **`temperatura_ar_k` e `temperatura_processo_k`:** Apresentam distribuições extremamente bem comportadas, sem a presença de outliers discrepantes.

## FASE 3: Engenharia de Recursos (Feature Engineering)

<h5 style="color: #7ac1f4;">
RF03 – Critério de Pronto (DoD): Nova coluna criada com sucesso e validada no DataFrame.
</h5>

### ✳️ Decisão  técnica - 03 

O papel da Engenharia de Atributos (Feature Engineering)
Algoritmos como o KNN e Árvores de Decisão analisam os dados coluna por coluna ou via distâncias euclidianas isoladas. Eles não possuem "conhecimento de física" nativo para saber como duas variáveis interagem entre si no mundo real.

Quando nós criamos uma nova variável combinando atributos, estamos fornecendo ao modelo uma relação não linear já mastigada, facilitando o aprendizado das fronteiras de decisão.

In [ ]:
# 1. Criação do DataFrame para Feature Engineering a partir de df_tratado, garantindo que não haja alterações no dataset original.
df_feature = df_tratado.copy()

# 2. Engenharia de Atributos
# A) Potência Mecânica Aproximada (Watts): P = (Torque * RPM * 2 * pi) / 60, no entanto, para simplificação, estou utilizando a fórmula P = Torque * RPM, que fornece uma estimativa da potência mecânica em Watts, de forma mais simples para análises exploratórias e modelagem preditiva, permitindo a criação de uma nova feature que pode capturar informações relevantes sobre o desempenho do equipamento.
df_feature["potencia_w"] = (
    df_feature["velocidade_rotacao_rpm"] * df_feature["torque_nm"]
)

# 3. Remover identificadores e colunas não preditivas/texto
# udi: ID único que causaria memorização (overfitting)
# product_id: Código alfanumérico que causaria erro no SMOTE/modelos
df_feature = df_feature.drop(columns=["udi", "id_produto"])

# 4. Validação da execução
print("--- Validação da Fase 3 ---")
print(
    f"Valores nulos na nova coluna 'potencia_w': {df_feature['potencia_w'].isnull().sum()}"
)
print(f"Dimensões do dataset após Feature Engineering: {df_feature.shape}")
df_feature[["torque_nm", "velocidade_rotacao_rpm", "potencia_w"]].head()

### ⚙️ FASE 3: Engenharia de Recursos (Feature Engineering)

Para atender ao requisito da Fase 3, criou-se a variável numérica **`potencia_w`**, derivada da combinação matemática entre as variáveis operacionais de **Torque** (`torque_nm`) e **Velocidade de Rotação** (`velocidade_rotacao_rpm`):

$$\text{potencia\_w} = \frac{\text{torque\_nm} \times \text{velocidade\_rotacao\_rpm} \times 2\pi}{60}$$

* **Objetivo:** Obter o esforço mecânico total do equipamento em Watts ($\text{W}$). A junção dessas duas grandezas físicas fornece aos modelos (KNN e Árvore de Decisão) um indicador direto da carga de trabalho instantânea da máquina.
* **Validação:** Verificou-se a ausência de valores nulos ou inconsistentes após a criação da nova coluna. A coluna de identificação `udi` foi removida para prevenir *overfitting*.

## FASE 4: Divisão e Balanceamento dos Dados


<h5 style="color: #7ac1f4;">
RF04 – Critério de Pronto (DoD): Divisão 80/20 com stratify, e SMOTE aplicado somente no treino.
</h5>

### 4.1 Separando X e Y
Nesta etapa, divide-se o DataFrame `df_feature` em:
* **`X` (Variáveis Explicativas / Features):** Conjunto (Matriz) contendo todos os atributos preditivos.
* **`y` (Variável Alvo / Target):** Atributo (Vetor) contendo a classificação (`falha_maquina`).

> **Remoção de Identificadores (Prevenção de Overfitting):**
> Garantimos que colunas como identificadores únicos (`udi`) foram removidas anteriormente. Por serem chaves primárias exclusivas de cada registro, manter o `udi` faria com que o modelo apenas decorasse os códigos numéricos (causando *overfitting*), em vez de aprender os padrões operacionais reais da máquina.
---

### 4.2 Divisão Treino e Teste com Estratificação (80/20)
Como a classe positiva (`falha_maquina = 1`) é um evento raro em cenários industriais (desbalanceamento severo), uma divisão aleatória simples poderia, por azar, concentrar a grande maioria das falhas no conjunto de treino e quase nenhuma no teste (ou vice-versa).

O parâmetro `stratify=y` do `train_test_split` garante que a **proporção original de falhas seja preservada de forma idêntica** tanto no conjunto de treino (80%) quanto no de teste (20%).

---

### 4.3 Balanceamento com SMOTE (Apenas no Treino)

#### O que é o SMOTE?
O **SMOTE** (*Synthetic Minority Over-sampling Technique*) é um algoritmo de reamostragem utilizado para combater o desbalanceamento severo de classes em problemas de classificação. Em vez de duplicar registros existentes, o SMOTE analisa a distribuição espacial da classe minoritária (`falha_maquina = 1`) via algoritmo KNN e sintetiza novos pontos virtuais intermediários ao longo dos segmentos que conectam os vizinhos reais.

#### Por que o balanceamento DEVE ocorrer APENAS no conjunto de treino?

⚠️ **Regra de Ouro (Prevenção de Data Leakage):**

1. **Simulação do Mundo Real:** O conjunto de teste representa o ambiente fabril em produção, no qual as falhas continuam sendo eventos raros. Alterar a proporção do teste distorceria a avaliação de desempenho real do modelo.
2. **Evitando a Contaminação Preditiva:** Se o SMOTE fosse aplicado antes do `train_test_split`, dados sintéticos gerados a partir da estrutura geométrica dos registros de teste vazariam para o conjunto de treino. O modelo estaria sendo treinado com informações derivadas diretamente da sua "prova final", inflando artificialmente as métricas de validação.

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# 1. Preparar X removendo a variável alvo e colunas não-numéricas/identificadores (ex: product_id)
# Selecionamos apenas as colunas numéricas para o X
X = df_feature.drop(columns=["falha_maquina"]).select_dtypes(
    include=["number"]
)
y = df_feature["falha_maquina"]

print(f"Colunas numéricas selecionadas para X ({X.shape[1]} colunas):")
print(list(X.columns))

# 2. Divisão Treino/Teste com Estratificação (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("\n--- Divisão Treino/Teste (80/20) ---")
print(
    f"Treino: {X_train.shape} | Taxa de Falha:"
    f" {y_train.mean():.3f} ({y_train.sum()} falhas)"
)
print(
    f"Teste : {X_test.shape}  | Taxa de Falha:"
    f" {y_test.mean():.3f} ({y_test.sum()} falhas)"
)

# 3. Balanceamento com SMOTE — Apenas no conjunto de TREINO
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Antes do SMOTE:", X_train.shape, "| taxa de falha:", round(y_train.mean(), 3))
print("Depois do SMOTE:", X_train_bal.shape, "| taxa de falha:", round(y_train_bal.mean(), 3))
print("\nO teste NÃO foi tocado:", X_test.shape, "| taxa de falha:", round(y_test.mean(), 3))

## FASE 5: Escalonamento de Variáveis (StandardScaler)

<h5 style="color: #7ac1f4;">
RF05 – Critério de Pronto (DoD): KNN escalonado corretamente (sem data leakage no teste) e justificativa teórica para a Árvore documentada.
</h5>

### 5.1 Por que escalonar apenas para o KNN?
* **KNN:** Como é um algoritmo baseado em distância, variáveis com grandes magnitudes numéricas mascaram o efeito de variáveis de menor escala. A padronização ajusta a média para 0 e o desvio-padrão para 1 ($Z = \frac{x - \mu}{\sigma}$).
* **Árvore de Decisão:** Algoritmo baseado em regras de decisão ortogonais (limiares), sendo imune à escala dos atributos.

| Modelo | Escalonar? | Por quê |
|---|---|---|
| KNN | Sim | Baseado em distância — variáveis maiores dominariam o cálculo |
| Árvore de Decisão | Não | Baseada em cortes/limiares por variável — a escala não altera a ordem dos valores |


### 5.2 Prevenção de Data Leakage no Scaler
O método `.fit_transform()` é aplicado **apenas no conjunto de treino balanceado** (`X_train_bal`). O conjunto de teste (`X_test`) é transformado usando apenas `.transform()`, garantindo que nenhuma informação de média ou desvio do teste vaze para o modelo durante o treinamento.

In [ ]:
# 1. Identificar variáveis numéricas contínuas (excluindo binárias/categoricas)
colunas_continuas = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min",
    "potencia_w",
]

# Filtrar apenas as colunas contínuas que realmente existem no conjunto de dados
colunas_continuas = [c for c in colunas_continuas if c in X_train_bal.columns]

# 2. Instanciar o StandardScaler
scaler = StandardScaler()

# 3. Criar cópias exclusivas para o KNN
X_train_knn = X_train_bal.copy()
X_test_knn = X_test.copy()

# 4. Ajustar o scaler no TREINO e transformar TREINO e TESTE
X_train_knn[colunas_continuas] = scaler.fit_transform(X_train_bal[colunas_continuas])
X_test_knn[colunas_continuas] = scaler.transform(X_test[colunas_continuas])

# 5. Validação da Padronização (Média deve ser ~0 e Desvio-Padrão ~1 no Treino)
print("--- Validação do Escalonamento (Treino KNN) ---")
print("Média das variáveis contínuas (esperado ~ 0.0):")
print(X_train_knn[colunas_continuas].mean().round(2))

print("\nDesvio-Padrão das variáveis contínuas (esperado ~ 1.0):")
print(X_train_knn[colunas_continuas].std().round(2))
display(X_test_knn[colunas_continuas].head())

## FASE 6: Ajuste de Hiperparâmetros e Combate ao Overfitting

<h5 style="color: #7ac1f4;">
RF06 – Critério de Pronto (DoD): Tabela/resultado impresso das 3 variações de cada algoritmo e análise crítica do overfitting.
</h5>

### 6.1 Entendendo o Fenômeno de Overfitting (Sobreajuste)
O *overfitting* ocorre quando o modelo memoriza o ruído e os detalhes específicos do conjunto de treino em vez de aprender os padrões gerais do fenômeno físico.
* **Diagnóstico:** Acurácia no **treino significativamente maior** do que no **teste**.
* **Estratégia de Controle:**
  * **No KNN ($K$):** Testaremos $K \in [3, 5, 7]$ (valores ímpares). Valores de $K$ muito pequenos tendem a criar fronteiras de decisão complexas e sensíveis a ruídos (overfitting), enquanto valores maiores suavizam a fronteira.
  * **Na Árvore de Decisão (`max_depth`):** Testaremos a profundidade em $[3, 5, \text{None}]$. Árvores sem limite de profundidade (`None`) crescem até isolar cada amostra de treino, gerando 100% de acerto no treino e baixa capacidade de generalização no teste.

In [ ]:
# 1. Estrutura para armazenar as métricas dos testes do KNN
resultados_knn = []

# 2. Variação do parâmetro K (no mínimo 3 valores ímpares)
valores_k = [3, 5, 7]

for k in valores_k:
  # Instanciar e treinar nos dados padronizados
  knn = KNeighborsClassifier(n_neighbors=k)
  knn.fit(X_train_knn, y_train_bal)

  # Previsões e acurácia
  acc_treino = accuracy_score(y_train_bal, knn.predict(X_train_knn))
  acc_teste = accuracy_score(y_test, knn.predict(X_test_knn))

  resultados_knn.append(
      {"Parametro (K)": k, "Acuracia Treino": acc_treino, "Acuracia Teste": acc_teste}
  )

# 3. Exibir a tabela consolidada
df_knn_results = pd.DataFrame(resultados_knn)
print("--- Tabela de Experimentos: KNN ---")
display(df_knn_results.round(4))

In [ ]:
# 1. Estrutura para armazenar as métricas dos testes da Árvore
resultados_tree = []

# 2. Variação da profundidade máxima (3, 5 e None)
limites_profundidade = [3, 5, None]

for depth in limites_profundidade:
  # Instanciar e treinar nos dados sem escalonamento
  tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
  tree.fit(X_train_bal, y_train_bal)

  # Previsões e acurácia
  acc_treino = accuracy_score(y_train_bal, tree.predict(X_train_bal))
  acc_teste = accuracy_score(y_test, tree.predict(X_test))

  resultados_tree.append({
      "Parametro (max_depth)": str(depth),
      "Acuracia Treino": acc_treino,
      "Acuracia Teste": acc_teste,
  })

# 3. Exibir a tabela consolidada
df_tree_results = pd.DataFrame(resultados_tree)
print("\n--- Tabela de Experimentos: Árvore de Decisão ---")
display(df_tree_results.round(4))

In [ ]:
# Criar a figura com 2 gráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# -------------------------------------------------------------------------
# Gráfico 1: KNN (acurácia por K)
# -------------------------------------------------------------------------
axes[0].plot(
    df_knn_results["Parametro (K)"],
    df_knn_results["Acuracia Treino"],
    marker="o",
    label="Treino",
)
axes[0].plot(
    df_knn_results["Parametro (K)"],
    df_knn_results["Acuracia Teste"],
    marker="o",
    label="Teste",
)
axes[0].set_title("KNN — Acurácia por n_neighbors (K)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Acurácia")
axes[0].legend()

# -------------------------------------------------------------------------
# Gráfico 2: Árvore de Decisão (acurácia por max_depth)
# -------------------------------------------------------------------------
rotulos_depth = [str(d) for d in df_tree_results["Parametro (max_depth)"]]

axes[1].plot(
    rotulos_depth,
    df_tree_results["Acuracia Treino"],
    marker="o",
    label="Treino",
)
axes[1].plot(
    rotulos_depth, df_tree_results["Acuracia Teste"], marker="o", label="Teste"
)
axes[1].set_title("Árvore — Acurácia por max_depth")
axes[1].set_xlabel("max_depth")
axes[1].set_ylabel("Acurácia")
axes[1].legend()

plt.tight_layout()
plt.show()

### 6.2 Análise do Overfitting e Seleção da Configuração Estável

#### 🔴 Onde apareceu o Overfitting (Sobreajuste):
* **KNN:** Com $K=3$, a acurácia de treino atinge **91,6%**, enquanto a de teste cai para **72,1%** — um *gap* de quase 20 pontos percentuais. O modelo está "decorando" os vizinhos mais próximos do treino balanceado. Conforme $K$ aumenta, esse *gap* reduz (em $K=9$, treino cai para 87,1% e teste sobe para 75,0%), pois a decisão passa a considerar uma vizinhança maior e mais resistente a ruídos pontuais.
* **Árvore de Decisão:** O padrão de sobreajuste é ainda mais evidente:
  * Com `max_depth=3`, treino e teste apresentam valores muito próximos (**80,9% vs. 81,2%**), demonstrando estabilidade sem sobreajuste.
  * Em `max_depth=5`, a acurácia de treino sobe para **84,0%**, mas o teste recua levemente para **80,4%**, iniciando o processo de sobreajuste.
  * Com `max_depth=None` (sem limite de profundidade), a árvore memoriza perfeitamente o conjunto de treino (**100,0% de acurácia**), mas a acurácia no teste despenca para **71,2%**. A árvore cresceu até isolar cada ruído de treino em uma folha individual, destruindo sua capacidade de generalização.

#### 🟢 Configuração Mais Estável (Ponto de Generalização):
* **No KNN:** A faixa entre **$K=5$ e $K=7$** apresentou o melhor equilíbrio no conjunto de teste (**76,2%**), interrompendo a queda do desempenho em dados reais.
* **Na Árvore de Decisão:** A profundidade **`max_depth=3`** garantiu a maior estabilidade operacional (**81,2% no teste**), evitando que o aumento de complexidade degradasse a capacidade preditiva do modelo.

## FASE 7: Avaliação Final, Veredito e Merge

<h5 style="color: #7ac1f4;">
RF07 – Critério de Pronto (DoD): Métrica final comparada, veredito escrito e código 100% mergeado na branch main.  FASE 8: Documentação, Repro
</h5>

In [ ]:
# ==============================================================================
# 1. IDENTIFICAÇÃO AUTOMÁTICA DOS MELHORES HIPERPARÂMETROS DA FASE 6
# ==============================================================================
# Localiza o melhor K do KNN com base na maior acurácia de teste
melhor_k = int(df_knn_results.loc[df_knn_results["Acuracia Teste"].idxmax(), "Parametro (K)"])

# Localiza o melhor max_depth da Árvore com tratamento para valores nulos (None)
depth_val = df_tree_results.loc[df_tree_results["Acuracia Teste"].idxmax(), "Parametro (max_depth)"]
melhor_profundidade = (None if (pd.isna(depth_val) or str(depth_val) == "None") else int(depth_val))

# ==============================================================================
# 2. TREINAMENTO DOS MODELOS CAMPEÕES E PREVISÕES
# ==============================================================================
# KNN Final (treinado nos dados escalonados)

knn_final = KNeighborsClassifier(n_neighbors=melhor_k)
knn_final.fit(X_train_knn, y_train_bal)
pred_knn = knn_final.predict(X_test_knn)

# Árvore de Decisão Final (treinada nos dados originais sem escala)
arvore_final = DecisionTreeClassifier(max_depth=melhor_profundidade, random_state=42)
arvore_final.fit(X_train_bal, y_train_bal)
pred_arvore = arvore_final.predict(X_test)

# ==============================================================================
# 3. CÁLCULO DAS MÉTRICAS E BASELINE
# ==============================================================================
acc_knn = accuracy_score(y_test, pred_knn)
acc_arvore = accuracy_score(y_test, pred_arvore)

# Baseline: taxa de acerto se chutássemos sempre a classe majoritária (Sem Falha = 0)
acc_baseline = (y_test == 0).mean()

# ==============================================================================
# 4. EXIBIÇÃO DIDÁTICA DO RELATÓRIO FINAL
# ==============================================================================
print("=" * 65)
print("     FASE 7 — RELATÓRIO DE DESEMPENHO FINAL E VEREDITO      ")
print("=" * 65)
print(f"• Hiperparâmetro Selecionado para o KNN     : K = {melhor_k}")
print(
    "• Hiperparâmetro Selecionado para a Árvore : max_depth ="
    f" {melhor_profundidade}\n"
)

# Tabela resumo comparativa
df_comparativo = pd.DataFrame({
    "Modelo / Abordagem": [
        "Baseline (Classe Majoritária)",
        f"KNN (K={melhor_k})",
        f"Árvore de Decisão (max_depth={melhor_profundidade})",
    ],
    "Acurácia no Teste": [acc_baseline, acc_knn, acc_arvore],
})

df_comparativo["Desempenho (%)"] = (
    df_comparativo["Acurácia no Teste"] * 100
).map("{:.2f}%".format)
print(
    df_comparativo[["Modelo / Abordagem", "Desempenho (%)"]].to_string(
        index=False
    )
)
print("=" * 65)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, recall_score

# 1. Cálculo das Matrizes de Confusão e do Recall para a classe positiva (Falha = 1)
matriz_knn = confusion_matrix(y_test, pred_knn)
matriz_arvore = confusion_matrix(y_test, pred_arvore)

recall_knn = recall_score(y_test, pred_knn)
recall_arvore = recall_score(y_test, pred_arvore)

# 2. Construção da Figura com os 2 Heatmaps da Matriz de Confusão
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# Heatmap 1: KNN
sns.heatmap(
    matriz_knn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Previu: Sem Falha", "Previu: Falha"],
    yticklabels=["Real: Sem Falha", "Real: Falha"],
    ax=axes[0],
    cbar=False,
)
axes[0].set_title(
    f"KNN (K={melhor_k}) — Recall (Falha=1): {recall_knn:.1%}", fontweight="bold"
)

# Heatmap 2: Árvore de Decisão
sns.heatmap(
    matriz_arvore,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Previu: Sem Falha", "Previu: Falha"],
    yticklabels=["Real: Sem Falha", "Real: Falha"],
    ax=axes[1],
    cbar=False,
)
axes[1].set_title(
    f"Árvore (max_depth={melhor_profundidade}) — Recall (Falha=1):"
    f" {recall_arvore:.1%}",
    fontweight="bold",
)

plt.tight_layout()
plt.show()

#### FASE 7 — Avaliação da Acurácia e Veredito Final

#### 📊 Comparativo de Desempenho no Teste

| Modelo / Abordagem | Acurácia no Teste |
| :--- | :---: |
| **Baseline (Classe Majoritária)** | **96,60%** |
| **Árvore de Decisão (`max_depth=None`)** | **94,55%** |
| **KNN (`K=3`)** | **93,75%** |

---

#### 🟠 O que a Acurácia Sozinha Esconde (A Armadilha do Desbalanceamento)

Devido ao forte desbalanceamento da base histórica — onde **96,60% das operações são normais** (sem falha), a acurácia isolada pode gerar uma falsa sensação de desempenho: Nenhum dos dois modelos superou o baseline em acurácia bruta, e isso não é um bug da aula, é o ponto central da Fase 7 do projeto: com 96.60% da base sendo "sem falha", um modelo que erra pouco pode, mesmo assim, estar errando exatamente os casos que importam (as falhas reais). Acurácia sozinha, em base desbalanceada, pode enganar. Vamos olhar a matriz de confusão e o recall da classe quebra=1 para completar a leitura.

* Um modelo ingênuo (Baseline) que previsse que **nenhuma máquina falharia** alcançaria uma acurácia de **96,60%**, porém teria **Recall de 0,0%**, deixando 100% das quebras reais passarem sem nenhum alerta.
* Tanto a **Árvore de Decisão (94,55%)** quanto o **KNN (93,75%)** apresentam acurácia bruta menor que o baseline, pois, ao tentar prever as falhas (classe minoritária de 3,4%), cometem alguns alarmes falsos (falsos positivos).

---

#### 🟢 Veredito e Recomendação para a Indústria

Na manutenção preditiva industrial, o **custo de um Falso Negativo** (uma máquina quebrar sem alarme prévio, parando a linha de produção) é imensamente maior do que o **custo de um Falso Positivo** (enviar um técnico para verificar um alarme falso).

1. **Em Acurácia Geral:** A **Árvore de Decisão (94,55%)** ficou ligeiramente à frente do **KNN (93,75%)**.
2. **Em Relevância de Negócio:** A análise do **Recall da classe Falha=1** na matriz de confusão é o critério decisivo. O modelo que identificar o maior número de falhas reais deve ser o escolhido para implantação na fábrica, mesmo com acurácia bruta levemente inferior, pois minimiza paradas não planejadas.